# 2. Análisis comparativo con MLflow

Este notebook toma las corridas ya registradas en MLflow y construye una comparación más rica que un simple ranking por métrica.

La idea es responder preguntas como:

- ¿El baseline clásico sigue siendo competitivo?
- ¿Conviene congelar o ajustar los embeddings preentrenados?
- ¿La cobertura del vocabulario ayuda a explicar diferencias de desempeño?
- ¿El mejor modelo también es una opción razonable para desplegar?

## Conexión con MLflow

Si tu servidor de tracking vive en otra máquina, define `MLFLOW_TRACKING_URI` antes de usar estas celdas.

El script `generate_report.py` automatiza una primera versión del análisis y deja los artefactos listos para anexarlos al informe.

In [ ]:
import os
import mlflow
import pandas as pd
from pathlib import Path

tracking_uri = os.getenv("MLFLOW_TRACKING_URI")
if tracking_uri:
    mlflow.set_tracking_uri(tracking_uri)

print("Tracking URI activo:", mlflow.get_tracking_uri())

In [ ]:
!python ../scripts/generate_report.py --experiment-name imdb-spanish-sentiment

## Revisar la tabla consolidada

La tabla siguiente permite comparar de manera rápida F1 y ROC-AUC en validación y prueba. La columna clave para elegir modelo suele ser `test_f1`, pero no debe leerse sola.

Por ejemplo:

- Si un modelo sube mucho en validación pero cae en prueba, puede haber sobreajuste.
- Si dos modelos empatan en F1, conviene mirar costo computacional y facilidad de despliegue.
- Si afinar embeddings mejora poco frente al modelo congelado, quizá no valga el costo extra.

In [ ]:
comparison = pd.read_csv(Path("../artifacts/reports/comparison.csv"))
comparison[["run_name", "model_family", "test_f1", "test_roc_auc", "val_f1", "val_roc_auc"]]

## Preguntas para redactar el análisis final

Usa los resultados para sustentar, no solo para enumerar:

1. ¿Por qué el baseline se comportó mejor o peor que la red neuronal?
2. ¿Qué embeddings mostraron mayor cobertura y cómo se reflejó eso en las métricas?
3. ¿Qué diferencia concreta produjo congelar vs ajustar los vectores?
4. ¿El mejor modelo es también el más práctico para producción?

La respuesta ideal conecta métricas, cobertura de vocabulario, capacidad del modelo y costo de entrenamiento.

In [ ]:
best_run = comparison.sort_values("test_f1", ascending=False).iloc[0]
best_run